## Setting Up Training, Validation, and Testing Data

In [32]:
import json
spkrs, txts, ner = [], [], []
with open("yes_self_mod.txt") as file:
    while True:
        speech_ln = file.readline()
        ner_ln = file.readline()
        while ner_ln and ner_ln[0] != "[":
            speech_ln += ner_ln
            ner_ln = file.readline()
        if not ner_ln:
            break
        ner_ln = ner_ln.replace("\'", "\"")
        splits = speech_ln.split(":", 1)
        if len(splits) != 2:
            continue
        speaker, text = splits
        if "Committee" not in speaker and len(speaker) > 1:
            spkrs.append(speaker.strip())
            txts.append(text.strip())
            ner.append(json.loads(ner_ln))

In [33]:
print(len(spkrs))

10453


In [3]:
data = list(zip(spkrs, txts, ner))
data[2]

('Tam Ma',
 "hi, good morning, mr. chairs and members. ma with health access california, the state healthcare consumer advocacy coalition. we worked very closely with representative becerra and his office to ensure passage of the affordable care act and we have the utmost confidence that he will continue working hard to ensure that all californians have access to quality and affordable health care and on a personal note, i grew up in mr. becerra's district, and my mother still resides there and i'm very honored to be here today in support of his nomination. thank you.",
 [{'start': 22, 'end': 28, 'label': 'PERSON'},
  {'start': 169, 'end': 176, 'label': 'PERSON'},
  {'start': 429, 'end': 436, 'label': 'PERSON'}])

In [4]:
training_data = {'classes': ['PERSON', 'SPEAKER'], 'annotations': []}
for i in data:
    temp_dict = {}
    temp_dict['text'] = i[1]
    temp_dict['entities'] = []
    for tag in i[2]:
      temp_dict['entities'].append((tag['start'], tag['end'], tag['label']))
    training_data['annotations'].append(temp_dict)

training_data['annotations'][0]

{'text': "thank you, speaker, and the members of the assembly. i'm steven, as you already heard the last time. as a korean-descent member of the community, when i hear that assemblyman matthew harper proposed to adjourn our session in memory of dr. sammy lee, who was a korean american, and myself, i first of all appreciate for that suggestion and i must add my few comments, as i know he was such a great role model, a hero to our community. i know he was well-known as a olympic gold medalist, as a physician, and he served his community for a long time. i met him several times when he came to the city of irvine in commemoration of korean american day. he's full of humor, wit, and knowledge and every time he speaks, people crack up with laughs. so such a wonderful human being. and then also, we recognize him as a very small physical stature, however, he was a giant to our korean american community. he continues to serve as a role model of our community. so this is very proper, and i appre

In [5]:
import spacy
from spacy.tokens import DocBin
from tqdm import tqdm

nlp = spacy.blank("en") # load a new spacy model

In [6]:
from spacy.util import filter_spans

def save_spacy_data(data, output):
    doc_bin = DocBin() # create a DocBin object
    entities_skipped = 0
    invalid_spans = 0
    missing_labels = 0

    for training_example in data:
        text = training_example['text']
        labels = training_example['entities']
        doc = nlp.make_doc(text)
        ents = []

        for start, end, label in labels:
            if not label:  # Skip entities with missing labels
                print(f"Skipping entity with missing label in text: {text}")
                missing_labels += 1
                continue

            span = doc.char_span(start, end, label=label, alignment_mode="contract")
            if span is None:
                print(f"Skipping invalid entity (None) in text: {text}")
                entities_skipped += 1
                continue

            # Check for leading or trailing whitespace in the span
            span_text = span.text
            if span_text != span_text.strip():  # If there is leading or trailing whitespace
                print(f"Skipping invalid entity span with whitespace: '{span_text}' in text: {text}")
                invalid_spans += 1
                continue

            ents.append(span)

        # Filter valid spans and assign them to the doc
        filtered_ents = filter_spans(ents)
        doc.ents = filtered_ents
        doc_bin.add(doc)

    print(f"Total entities skipped: {entities_skipped}")
    print(f"Total invalid entity spans (whitespace): {invalid_spans}")
    print(f"Total entities with missing labels: {missing_labels}")
    
    doc_bin.to_disk(output)

In [7]:
for annotation in training_data["annotations"]:
    for start, end, label in annotation["entities"]:
        span = annotation["text"][start:end]
        if span != span.strip():  # If there is leading or trailing whitespace
            print(f"Invalid span: '{span}' in text: '{annotation['text']}'")

Invalid span: ' laurie aja' in text: 'thank you for the opportunity. my name is kristi, i'm excited to be here to provide some feedback, so thank you for the opportunity. i'm the owner and co-founder of kiva confections, we're a chocolate manufacturing company, edibles company, located in oakland and we also do distribution throughout the state. i'm here as a member of ccma and ccia, the california cannabis industry association. i sit on the board of directors there and i also chair the manufacturing committee. the ccia has worked with this legislature to provide feedback on both merca and prop 64 and we're excited to continue to do that with you, to work together and provide some collaboration so we can make sure we get the system done right. so one of the first items is the importance of a phase in with the licensing program, so laurie ajax mentioned this earlier as well,  but the ability for a licensed business to also be able to work with unlicensed businesses for a set amount of t

In [8]:
import random

random.seed(160)
random.shuffle(training_data['annotations'])

training_ratio = 0.8
validation_ratio = 0.1
test_ratio = 1 - training_ratio - validation_ratio

train_split_index = int(len(training_data['annotations']) * training_ratio)
dev_split_index = train_split_index + int(len(training_data['annotations']) * validation_ratio)

TRAINING_DATA = training_data['annotations'][:train_split_index]
DEV_DATA = training_data['annotations'][train_split_index:dev_split_index]
TEST_DATA = training_data['annotations'][dev_split_index:]

print(len(TRAINING_DATA))
print(len(DEV_DATA))
print(len(TEST_DATA))

save_spacy_data(TRAINING_DATA, "train.spacy")
save_spacy_data(DEV_DATA, "dev.spacy")
save_spacy_data(TEST_DATA, "test.spacy")

8362
1045
1046
Skipping invalid entity (None) in text: good afternoon, my name is francisco. i work for dole in the harvesting department. and we've come here to support mr. hall and miss montgomery.  for the new positions that they are positioning themselves for. gracias.
Skipping invalid entity (None) in text: Good afternoon. My name is Leticia Munoz. And I come from Madeira, California. And I work in the Harvesting of mandarins.  And I come to support Senator Hall and Ms. Montgomery.
Skipping invalid entity (None) in text: Good afternoon. My name is Leticia Munoz. And I come from Madeira, California. And I work in the Harvesting of mandarins.  And I come to support Senator Hall and Ms. Montgomery.
Skipping invalid entity span with whitespace: 'usha  ' in text: good morning, usha  mutschler with the california state sheriffs association, in strong support.
Skipping invalid entity (None) in text: hello my name is hennigan desiree pittsburgh and i'm here today on behalf of sacramento w

## Commands to Run to Train Model

python -m spacy init fill-config base_config.cfg config.cfg

python -m spacy train config.cfg --output ./MODELTYPE --paths.train ./train.spacy --paths.dev ./dev.spacy 

## Run if data has errors

python -m spacy debug data config.cfg --paths.train ./train.spacy --paths.dev ./dev.spacy

## Running the Model

In [9]:
spacy.load("en_core_web_lg")
spacy.load("en_core_web_trf")

c:\Users\ktann\AppData\Local\Programs\Python\Python312\Lib\site-packages\thinc\shims\pytorch.py:261: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torc

In [18]:
# Fine tuning on trf model
nlp_ner = spacy.load("trf/model-best")
# doc = nlp_ner("Hi, good morning, Mr. Chairs and Members. Choi with Health Access California, the State Healthcare Consumer Advocacy Coalition. We worked very closely with Representative Becerra and his office to ensure passage of the Affordable Care Act and we have the utmost confidence that he will continue working hard to ensure that all Californians have access to quality and affordable health care and on a personal note, I grew up in Mr. Becerra's district, and my mother still resides there and I'm very honored to be here today in support of his nomination. Thank you.")
# doc = nlp_ner("Mr Chair And Members, Rand Martin On Behalf Of Consortium Management Group And Caliva. It'S A Large Dispensary Manufacturing And Cultivation Operation In San Jose. First, I Want To Echo The Compliments Paid To This Committee And To The Members Of The Regulatory Agencies For All The Hard Work That'S Being Done. Caliva Comes From A History, With Its Leadership Of A Highly Regulated Industries In Other Fields Under Medicaid And Medicare. So They Have A Lot Of Respect For Strong Regulation. They Would Like To See Strong Regulation Here. Regulation Is Not Always Agreed Upon By Even The People Sitting At This Table. And Mr Conway'S Comments Bring To Mind One Point About Where Should Cannabis Be Cultivated. There Are Some People Who Believe It Should Only Happen In Rural Areas. And Some Who Believe That Enclosed, Odor-Free Urban Cultivation Is A Reasonable Approach As Well. So I Respect The Fact That The Regulatory Agencies Have Some Work To Do With The Industry, And Figuring Out Exactly How To Thread These Various Needles. I Do Want To Point Out A Few Things, Just Relative To The Very Specifics Of Regulations, And Won'T Get Into Any Detail. I Do Want To Echo Some Of The Things That Ms Jenkins Said About Aligning Amma With Mcrsa. We Agree With The Points That She Made, We Would Add One Additional One. And That Is The Issue Of Ownership. Ownership In Mcrsa, At Least Where Publicly Traded, Is At 5%, Under Amma, It'S 20%. We'D Like To See Alignment There As Well. Another Ownership Issue, That Is Something That You May Have To Deal With This Year In Order To Be Helpful To Some Of Us In The Industry, Is How Quickly You Can Become A For-Profit Entity. Mcrsa Anticipates That Happening January 1, 2018. Businesses That Want To Get Geared Up For January 1St, Who Are Looking For Investments, Need To Know That They Can Transfer To A For-Profit Ownership Structure Earlier Than January 1St. Again, Anticipated On January 1St, But If We Can Get It Earlier, Perhaps As An Element In The Budget Trailer Bill This Year, That Would Be Hugely Helpful To A Lot Of Entities Out There That Are Looking Forward To Converting. The Only Other Point I Want To Make, And One That Has Not Been Raised At All This Evening, I'Ve Certainly Not Heard It Discussed Much, Is What It Is The Department Of Public Health Expectations Relative To Implementation Of Its Version Of The Food, Drug, And Cosmetic Act That Applies To Cannabis. For Those Who Have Spent Time With The Sherman Act In Years Past Know That There Are Issues In Which People Get Mired. The Core Issue Here For Us Is What Is Considered Misbranding Of A Product? Is A Product Misbranded If You Make Therapeutic, Beneficial Claims About The Product That Cannot Be Backed Up By The Science? Course We Find Ourselves Caught Between A Rock And A Hard Place, Because The Science Is Not There, Because It'S Been Illegal. And So Nobody Has Been Able To Study Some Of The Therapeutic Benefits. There Is An Operation Down At Uc San Diego That Has Made Some Initial Overtures In That Regard, But They Have A Long Way To Go. It'S Going To Be Very Important For The Department Of Public Health And Others Who Are Enforcing The Sherman Act To Be More Liberal In Their Interpretation Of What Therapeutic Benefits Are. Otherwise It Could Be Used As A Hammer Against The People Who Are Actually Seeking Licensure And Seeking To Turn This Into A Well-Regulated Business. It Deserves A Lot Of Attention, Not At This Time. But Certainly Hope That The Legislature Will Spend Some Time Looking At This Issue Over The Next Few Months. Thank You. Rand Martin")
doc = nlp_ner("Hi. Good morning, Senator Wieckowski, Senator Hill, Chair Nichols. My name is is Ryan Kenny, I'm with Clean Energy.")

colors = {"PERSON": "#F67DE3", "SPEAKER": "#7DF6D9"}
options = {"colors": colors} 

spacy.displacy.render(doc, style="ent", options= options, jupyter=True)

In [19]:
# Fine tuning on lg model
nlp_ner = spacy.load("large/model-best")
# doc = nlp_ner("Hi, good morning, Mr. Chairs and Members. Choi with Health Access California, the State Healthcare Consumer Advocacy Coalition. We worked very closely with Representative Becerra and his office to ensure passage of the Affordable Care Act and we have the utmost confidence that he will continue working hard to ensure that all Californians have access to quality and affordable health care and on a personal note, I grew up in Mr. Becerra's district, and my mother still resides there and I'm very honored to be here today in support of his nomination. Thank you.")
# doc = nlp_ner("Mr Chair And Members, Rand Martin On Behalf Of Consortium Management Group And Caliva. It'S A Large Dispensary Manufacturing And Cultivation Operation In San Jose. First, I Want To Echo The Compliments Paid To This Committee And To The Members Of The Regulatory Agencies For All The Hard Work That'S Being Done. Caliva Comes From A History, With Its Leadership Of A Highly Regulated Industries In Other Fields Under Medicaid And Medicare. So They Have A Lot Of Respect For Strong Regulation. They Would Like To See Strong Regulation Here. Regulation Is Not Always Agreed Upon By Even The People Sitting At This Table. And Mr Conway'S Comments Bring To Mind One Point About Where Should Cannabis Be Cultivated. There Are Some People Who Believe It Should Only Happen In Rural Areas. And Some Who Believe That Enclosed, Odor-Free Urban Cultivation Is A Reasonable Approach As Well. So I Respect The Fact That The Regulatory Agencies Have Some Work To Do With The Industry, And Figuring Out Exactly How To Thread These Various Needles. I Do Want To Point Out A Few Things, Just Relative To The Very Specifics Of Regulations, And Won'T Get Into Any Detail. I Do Want To Echo Some Of The Things That Ms Jenkins Said About Aligning Amma With Mcrsa. We Agree With The Points That She Made, We Would Add One Additional One. And That Is The Issue Of Ownership. Ownership In Mcrsa, At Least Where Publicly Traded, Is At 5%, Under Amma, It'S 20%. We'D Like To See Alignment There As Well. Another Ownership Issue, That Is Something That You May Have To Deal With This Year In Order To Be Helpful To Some Of Us In The Industry, Is How Quickly You Can Become A For-Profit Entity. Mcrsa Anticipates That Happening January 1, 2018. Businesses That Want To Get Geared Up For January 1St, Who Are Looking For Investments, Need To Know That They Can Transfer To A For-Profit Ownership Structure Earlier Than January 1St. Again, Anticipated On January 1St, But If We Can Get It Earlier, Perhaps As An Element In The Budget Trailer Bill This Year, That Would Be Hugely Helpful To A Lot Of Entities Out There That Are Looking Forward To Converting. The Only Other Point I Want To Make, And One That Has Not Been Raised At All This Evening, I'Ve Certainly Not Heard It Discussed Much, Is What It Is The Department Of Public Health Expectations Relative To Implementation Of Its Version Of The Food, Drug, And Cosmetic Act That Applies To Cannabis. For Those Who Have Spent Time With The Sherman Act In Years Past Know That There Are Issues In Which People Get Mired. The Core Issue Here For Us Is What Is Considered Misbranding Of A Product? Is A Product Misbranded If You Make Therapeutic, Beneficial Claims About The Product That Cannot Be Backed Up By The Science? Course We Find Ourselves Caught Between A Rock And A Hard Place, Because The Science Is Not There, Because It'S Been Illegal. And So Nobody Has Been Able To Study Some Of The Therapeutic Benefits. There Is An Operation Down At Uc San Diego That Has Made Some Initial Overtures In That Regard, But They Have A Long Way To Go. It'S Going To Be Very Important For The Department Of Public Health And Others Who Are Enforcing The Sherman Act To Be More Liberal In Their Interpretation Of What Therapeutic Benefits Are. Otherwise It Could Be Used As A Hammer Against The People Who Are Actually Seeking Licensure And Seeking To Turn This Into A Well-Regulated Business. It Deserves A Lot Of Attention, Not At This Time. But Certainly Hope That The Legislature Will Spend Some Time Looking At This Issue Over The Next Few Months. Thank You. Rand Martin")
doc = nlp_ner("Hi. Good morning, Senator Wieckowski, Senator Hill, Chair Nichols. My name is is Ryan Kenny, I'm with Clean Energy.")

colors = {"PERSON": "#F67DE3", "SPEAKER": "#7DF6D9"}
options = {"colors": colors} 

spacy.displacy.render(doc, style="ent", options= options, jupyter=True)

ℹ Saving to output directory: .
ℹ Using CPU

=========================== Initializing pipeline ===========================
✔ Initialized pipeline

============================= Training pipeline =============================
ℹ Pipeline: ['tok2vec', 'ner']
ℹ Initial learn rate: 0.001
E    #       LOSS TOK2VEC  LOSS NER  ENTS_F  ENTS_P  ENTS_R  SCORE 
---  ------  ------------  --------  ------  ------  ------  ------
  0       0          0.00     58.80    0.42    0.29    0.71    0.00
  0     200        294.65   2315.47   39.67   51.12   32.41    0.40
  0     400         58.02    523.39   67.68   79.39   58.97    0.68
  0     600         84.65    346.31   71.27   77.39   66.05    0.71
  0     800      57060.65    673.87   72.34   81.53   65.02    0.72
  0    1000        168.27    312.10   72.13   71.29   72.99    0.72
  0    1200        252.56    343.36   75.29   82.19   69.45    0.75
  0    1400        292.75    303.83   78.07   87.09   70.74    0.78
  0    1600        397.63    444.99   79.47   87.30   72.93    0.79
  0    1800        525.79    423.83   78.41   81.63   75.43    0.78
  0    2000        539.77    387.57   79.01   87.32   72.15    0.79
  0    2200        546.69    424.50   81.58   87.60   76.33    0.82
  0    2400       1005.44    440.16   78.62   87.98   71.06    0.79
  0    2600        524.86    362.36   79.64   80.45   78.84    0.80
  0    2800       2185.30    540.05   82.65   86.78   78.91    0.83
  0    3000       1733.65    509.57   79.43   83.33   75.88    0.79
  0    3200       1202.15    602.72   82.97   91.61   75.82    0.83
  1    3400       1799.39    644.03   81.30   87.71   75.76    0.81
  1    3600       1457.56    540.62   83.04   87.87   78.71    0.83
  1    3800       2803.09    532.99   82.50   81.07   83.99    0.83
  1    4000       2092.27    560.07   85.57   88.80   82.57    0.86
  1    4200       1950.88    570.28   85.70   88.33   83.22    0.86
  1    4400       2225.18    550.81   81.13   84.89   77.68    0.81
  1    4600       1495.43    576.68   84.80   88.25   81.61    0.85
  1    4800       3684.94    568.02   86.25   89.21   83.47    0.86
  1    5000       3742.78    564.02   85.76   87.96   83.67    0.86
  2    5200       2562.12    389.06   85.75   91.47   80.71    0.86
  2    5400       3228.79    482.89   87.31   87.93   86.69    0.87
  2    5600       1974.89    372.96   85.79   91.48   80.77    0.86
  2    5800       2781.27    407.32   87.25   88.76   85.79    0.87
  2    6000       3128.35    408.76   87.25   88.02   86.50    0.87
  2    6200       2678.27    415.96   87.38   89.87   85.02    0.87
  2    6400       4196.22    476.89   87.76   90.11   85.53    0.88
  2    6600       7293.05    457.65   84.88   81.89   88.10    0.85
  3    6800       2308.35    363.94   87.06   87.57   86.56    0.87
  3    7000       4433.50    331.01   87.80   88.61   87.01    0.88
  3    7200      12976.48    418.94   86.50   87.97   85.08    0.86
  3    7400       3949.63    349.79   86.40   91.15   82.12    0.86
  3    7600       6808.20    345.11   86.57   87.98   85.21    0.87
  3    7800       5879.76    362.67   87.38   86.61   88.17    0.87
  3    8000       8337.27    371.74   87.79   90.68   85.08    0.88
  3    8200       4749.60    384.96   87.46   89.34   85.66    0.87
  4    8400       3975.29    335.27   87.86   90.69   85.21    0.88
  4    8600      11039.22    290.33   86.78   90.50   83.34    0.87
  4    8800       5372.67    325.99   87.22   88.02   86.43    0.87
  4    9000       5011.26    337.69   87.34   88.67   86.05    0.87
  4    9200      14056.25    336.88   86.78   87.01   86.56    0.87
  4    9400       7132.89    344.32   88.47   91.78   85.40    0.88
  4    9600       5388.89    301.30   87.50   92.17   83.28    0.88
  4    9800      11965.58    457.10   88.34   90.05   86.69    0.88
  5   10000       4089.47    308.86   87.20   86.37   88.04    0.87
  5   10200       7503.19    261.94   85.77   85.50   86.05    0.86
  5   10400       4053.93    270.77   86.13   85.21   87.07    0.86
  5   10600       8833.95    289.48   85.38   86.68   84.12    0.85
  5   10800       6651.36    330.37   88.03   89.34   86.75    0.88
  5   11000       4788.48    281.38   87.42   89.11   85.79    0.87

## Testing the Model


In [35]:
spkrs, txts, ner = [], [], []
with open("yes_self_mod.txt") as file:
    while True:
        speech_ln = file.readline()
        ner_ln = file.readline()
        while ner_ln and ner_ln[0] != "[":
            speech_ln += ner_ln
            ner_ln = file.readline()
        if not ner_ln:
            break
        ner_ln = ner_ln.replace("\'", "\"")
        splits = speech_ln.split(":", 1)
        if len(splits) != 2:
            continue
        speaker, text = splits
        if "Committee" not in speaker and len(speaker) > 1:
            spkrs.append(speaker.strip())
            txts.append(text.strip())
            ner.append(json.loads(ner_ln))
print(len(spkrs))

10453


In [39]:
print(spkrs[0])
print(txts[0])
print(ner[0])

Steven Choi
thank you, speaker, and the members of the assembly. i'm steven, as you already heard the last time. as a korean-descent member of the community, when i hear that assemblyman matthew harper proposed to adjourn our session in memory of dr. sammy lee, who was a korean american, and myself, i first of all appreciate for that suggestion and i must add my few comments, as i know he was such a great role model, a hero to our community. i know he was well-known as a olympic gold medalist, as a physician, and he served his community for a long time. i met him several times when he came to the city of irvine in commemoration of korean american day. he's full of humor, wit, and knowledge and every time he speaks, people crack up with laughs. so such a wonderful human being. and then also, we recognize him as a very small physical stature, however, he was a giant to our korean american community. he continues to serve as a role model of our community. so this is very proper, and i app

In [36]:
doc = nlp_ner("Hi. Good morning, Senator Wieckowski, Senator Hill, Chair Nichols. My name is is Ryan Kenny, I'm with Clean Energy.")

# Extract labeled data
labeled_data = [(ent.text, ent.label_) for ent in doc.ents]

# Print the results
print("Labeled entities:")
for text, label in labeled_data:
    print(f"Text: {text}, Label: {label}")

Labeled entities:
Text: Wieckowski, Label: PERSON
Text: Hill, Label: PERSON
Text: Nichols, Label: PERSON
Text: Ryan Kenny, Label: SPEAKER


In [59]:
def testingModel(model):
    tp_person = 0
    fp_person = 0
    fn_person = 0
    tp_speaker = 0
    fp_speaker = 0
    fn_speaker = 0

    nlp = spacy.load(model)

    for text, labels in zip(txts, ner):
        doc = nlp(text)
        labeled_data = [(ent.text.strip().lower(), ent.label_) for ent in doc.ents]

        for label in labels:
            name = text[label["start"]:label["end"]].strip().lower()

            correct = False
            for i in labeled_data:
                if name == i[0] and label["label"] == i[1]:
                    correct = True
                    labeled_data.remove(i)  # Remove the matched entity
                    break  # Exit the loop after a match is found

            if correct:
                if label["label"] == 'SPEAKER':
                    tp_speaker += 1
                elif label["label"] == 'PERSON':
                    tp_person += 1
            else:
                if label["label"] == 'SPEAKER':
                    fn_speaker += 1  # Missed speaker
                elif label["label"] == 'PERSON':
                    fn_person += 1  # Missed person

        for i in labeled_data:
            if i[1] == 'SPEAKER':
                fp_speaker += 1
            elif i[1] == 'PERSON':
                fp_person += 1

    precision_person = tp_person / (tp_person + fp_person)
    recall_person = tp_person / (tp_person + fn_person)
    f1_person = 2 * (precision_person * recall_person) / (precision_person + recall_person)

    precision_speaker = tp_speaker / (tp_speaker + fp_speaker)
    recall_speaker = tp_speaker / (tp_speaker + fn_speaker)
    f1_speaker = 2 * (precision_speaker * recall_speaker) / (precision_speaker + recall_speaker)

    # Print results
    print(f"PERSON Precision: {precision_person:.4f}")
    print(f"PERSON Recall: {recall_person:.4f}")
    print(f"PERSON F1: {recall_person:.4f}")
    print(f"SPEAKER Precision: {precision_speaker:.4f}")
    print(f"SPEAKER Recall: {recall_speaker:.4f}")
    print(f"SPEAKER F1: {recall_speaker:.4f}")
    print(f"Overall Precision: {((precision_person + precision_speaker) / 2):.4f}")
    print(f"Overall Recall: {((recall_person + recall_speaker) / 2):.4f}")
    print(f"Overall F1: {((f1_person + f1_speaker) / 2):.4f}")

    return {"person_precision": precision_person, "person_recall": recall_person, "person_f1": f1_person, "speaker_precision": precision_speaker, "speaker_recall": recall_speaker, "speaker_f1": f1_speaker}

In [60]:
print("Large Model")
large_results = testingModel("large/model-best")

print("\nTRF Model")
trf_results = testingModel("trf/model-best")

Large Model
PERSON Precision: 0.8832
PERSON Recall: 0.7177
PERSON F1: 0.7177
SPEAKER Precision: 0.9329
SPEAKER Recall: 0.9779
SPEAKER F1: 0.9779
Overall Precision: 0.9080
Overall Recall: 0.8478
Overall F1: 0.8734

TRF Model
PERSON Precision: 0.9284
PERSON Recall: 0.7898
PERSON F1: 0.7898
SPEAKER Precision: 0.9649
SPEAKER Recall: 0.9742
SPEAKER F1: 0.9742
Overall Precision: 0.9466
Overall Recall: 0.8820
Overall F1: 0.9115


## Running Model with Gradio

In [61]:
import gradio as gr
from spacy import displacy

In [62]:
def title_case(text):
    return " ".join(map(lambda w: w.title(), text.split()))

def get_persons(doc):
    persons = []
    for ent in doc.ents:
        if ent.label_ == "PERSON":
            persons.append(ent)
    return persons

def make_persons_only(doc):
    doc.set_ents(get_persons(doc))

# displacy.render(doc, style="ent")

In [ ]:
# nlp = spacy.load("en_core_web_trf")
nlp_ner = spacy.load("trf/model-best")
# nlp_ner = spacy.load("large/model-best")

colors = {"PERSON": "#4169E1", "SPEAKER": "#FF0000"}
options = {"colors": colors} 

def text_analysis(text):
    text = title_case(text)

    doc = nlp_ner(text)
    # make_persons_only(doc)
    
    # html = displacy.render(doc, style="ent", page=True, jupyter=False, options=options)
    html = spacy.displacy.render(doc, style="ent", page=True, jupyter=False, options=options)
    html = (
        "<div style='max-width:100%; max-height:360px; overflow:auto'>"
        + html
        + "</div>"
    )

    return html, #pos_tokens, pos_count, html

demo = gr.Interface(
    fn=text_analysis,
    inputs=[gr.Textbox(placeholder="Enter sentence here...")],
    outputs=[gr.HTML()],
    examples=[
        ["Hi. Good morning, Senator Wieckowski, Senator Hill, Chair Nichols. My name is is Ryan Kenny, I'm with Clean Energy."],
        ["My friend atey azimith likes cats. My name is hans and my favorite food is turducken. I'm also partial to clementines. I'm also partial to carol."],
    ],
)

demo.launch(debug=True, share=True)

* Running on local URL:  http://127.0.0.1:7860

Could not create share link. Missing file: c:\Users\ktann\AppData\Local\Programs\Python\Python312\Lib\site-packages\gradio\frpc_windows_amd64_v0.3. 

Please check your internet connection. This can happen if your antivirus software blocks the download of this file. You can install manually by following these steps: 

1. Download this file: https://cdn-media.huggingface.co/frpc-gradio-0.3/frpc_windows_amd64.exe
2. Rename the downloaded file to: frpc_windows_amd64_v0.3
3. Move the file to this location: c:\Users\ktann\AppData\Local\Programs\Python\Python312\Lib\site-packages\gradio


Keyboard interruption in main thread... closing server.
